# NBA Breakout Analysis
**Author:** Mike Zhang  
**Date:** 2026-03-19  
**Purpose:** Identify players who broke out in the second half of seasons on non-playoff teams and examine whether their post all-star BPM predicts their following season BPM

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [35]:
# Pull post all-star stats for 2023-24
post_allstar = leaguedashplayerstats.LeagueDashPlayerStats(
    season='2023-24',
    season_segment_nullable='Post All-Star',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

# Pull pre all-star stats for 2023-24
pre_allstar = leaguedashplayerstats.LeagueDashPlayerStats(
    season='2023-24',
    season_segment_nullable='Pre All-Star',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

# Pull following full season stats (2024-25)
following_season = leaguedashplayerstats.LeagueDashPlayerStats(
    season='2024-25',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

print("Post all-star shape:", post_allstar.shape)
print("Pre all-star shape:", pre_allstar.shape)
print("Following season shape:", following_season.shape)

Post all-star shape: (518, 79)
Pre all-star shape: (540, 79)
Following season shape: (569, 79)


In [57]:
# Columns to keep from pre and post all-star splits
split_cols = ['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 
              'GP', 'MIN', 'PIE', 'USG_PCT']

# Columns to keep from following season
following_cols = ['PLAYER_ID',  'PIE']

# Trim each dataframe
post = post_allstar[split_cols].copy()
pre = pre_allstar[split_cols].copy()
following = following_season[following_cols].copy()

print(post.head())
print(pre.head())
print(following.head())

   PLAYER_ID    PLAYER_NAME     TEAM_ID TEAM_ABBREVIATION   AGE  GP   MIN  \
0    1630639    A.J. Lawson  1610612742               DAL  23.0  15   5.4   
1    1631260       AJ Green  1610612749               MIL  24.0  20  13.9   
2    1631100     AJ Griffin  1610612737               ATL  20.0   2  19.1   
3     203932   Aaron Gordon  1610612743               DEN  28.0  24  30.9   
4    1628988  Aaron Holiday  1610612745               HOU  27.0  27  13.2   

     PIE  USG_PCT  
0  0.081    0.204  
1  0.059    0.134  
2  0.014    0.204  
3  0.110    0.176  
4  0.066    0.168  
   PLAYER_ID    PLAYER_NAME     TEAM_ID TEAM_ABBREVIATION   AGE  GP   MIN  \
0    1630639    A.J. Lawson  1610612742               DAL  23.0  27   8.5   
1    1631260       AJ Green  1610612749               MIL  24.0  36   9.3   
2    1631100     AJ Griffin  1610612737               ATL  20.0  18   7.4   
3     203932   Aaron Gordon  1610612743               DEN  28.0  49  31.7   
4    1628988  Aaron Holiday  161

In [58]:
# Step 1: Apply post all-star inclusion criteria
# - At least 15 games played
# - At least 25 MPG
# - Age 28 or younger

post_filtered = post[
    (post['GP'] >= 15) & 
    (post['MIN'] >= 25) &
    (post['AGE'] <= 28)
].copy()

# Step 2: Apply pre all-star inclusion criteria
# - At least 20 games played
# - At least 10 MPG

pre_filtered = pre[
    (pre['GP'] >= 20) & 
    (pre['MIN'] >= 10)
].copy()

print("Post all-star candidates:", len(post_filtered))
print("Pre all-star candidates:", len(pre_filtered))

Post all-star candidates: 109
Pre all-star candidates: 355


In [60]:
# Merge pre and post all-star data on PLAYER_ID
merged = post_filtered.merge(
    pre_filtered,
    on='PLAYER_ID',
    suffixes=('_post', '_pre')
)

# Apply 40% minutes jump criterion
merged = merged[
    (merged['MIN_post'] - merged['MIN_pre']) / merged['MIN_pre'] >= 0.3
].copy()

print("Players meeting all criteria:", len(merged))
print(merged[['PLAYER_NAME_post', 'MIN_pre', 'MIN_post', 'PIE_post', 'PIE_pre']].to_string())

Players meeting all criteria: 13
     PLAYER_NAME_post  MIN_pre  MIN_post  PIE_post  PIE_pre
2       Amen Thompson     19.2      26.5     0.134    0.107
3         Amir Coffey     17.9      25.0     0.050    0.067
8         Ayo Dosunmu     25.5      37.5     0.092    0.077
20      Corey Kispert     22.5      32.4     0.082    0.085
32   Donte DiVincenzo     25.0      37.5     0.094    0.101
35         GG Jackson     19.4      31.4     0.090    0.105
37        Gradey Dick     15.3      28.8     0.060    0.057
63     Jordan Goodwin     14.6      29.6     0.100    0.095
66  Julian Champagnie     16.6      25.9     0.070    0.066
70         Keon Ellis     11.5      25.0     0.080    0.051
72        Kris Murray     13.1      32.2     0.055    0.067
83      Miles McBride     12.4      31.0     0.079    0.080
98    Trey Murphy III     25.8      34.0     0.118    0.101


In [61]:
# Filter for "tanking" teams in post all-star data
from nba_api.stats.endpoints import leaguedashteamstats

pre_allstar_teams = leaguedashteamstats.LeagueDashTeamStats(
    season='2023-24',
    season_segment_nullable='Pre All-Star'
).get_data_frames()[0]

print(pre_allstar_teams[['TEAM_NAME', 'TEAM_ID', 'GP', 'W', 'L', 'W_PCT']].to_string())

                 TEAM_NAME     TEAM_ID  GP   W   L  W_PCT
0            Atlanta Hawks  1610612737  55  24  31  0.436
1           Boston Celtics  1610612738  55  43  12  0.782
2            Brooklyn Nets  1610612751  54  21  33  0.389
3        Charlotte Hornets  1610612766  54  13  41  0.241
4            Chicago Bulls  1610612741  55  26  29  0.473
5      Cleveland Cavaliers  1610612739  53  36  17  0.679
6         Dallas Mavericks  1610612742  55  32  23  0.582
7           Denver Nuggets  1610612743  55  36  19  0.655
8          Detroit Pistons  1610612765  54   8  46  0.148
9    Golden State Warriors  1610612744  53  27  26  0.509
10         Houston Rockets  1610612745  54  24  30  0.444
11          Indiana Pacers  1610612754  56  31  25  0.554
12             LA Clippers  1610612746  53  36  17  0.679
13      Los Angeles Lakers  1610612747  56  30  26  0.536
14       Memphis Grizzlies  1610612763  56  20  36  0.357
15              Miami Heat  1610612748  55  30  25  0.545
16         Mil

In [62]:
tanking_teams = pre_allstar_teams.nsmallest(12, 'W_PCT')[['TEAM_NAME', 'TEAM_ID', 'W_PCT']].copy()
print(tanking_teams.sort_values('W_PCT').to_string())

                 TEAM_NAME     TEAM_ID  W_PCT
8          Detroit Pistons  1610612765  0.148
29      Washington Wizards  1610612764  0.167
26       San Antonio Spurs  1610612759  0.200
3        Charlotte Hornets  1610612766  0.241
24  Portland Trail Blazers  1610612757  0.278
27         Toronto Raptors  1610612761  0.345
14       Memphis Grizzlies  1610612763  0.357
2            Brooklyn Nets  1610612751  0.389
0            Atlanta Hawks  1610612737  0.436
10         Houston Rockets  1610612745  0.444
28               Utah Jazz  1610612762  0.464
4            Chicago Bulls  1610612741  0.473


In [63]:
# Filter merged players to only tanking teams
merged_tanking = merged[
    merged['TEAM_ID_post'].isin(tanking_teams['TEAM_ID'])
].copy()

print("Players on tanking teams:", len(merged_tanking))
print(merged_tanking[['PLAYER_NAME_post', 'TEAM_ID_post', 'MIN_pre', 'MIN_post', 'PIE_post']].to_string())

Players on tanking teams: 8
     PLAYER_NAME_post  TEAM_ID_post  MIN_pre  MIN_post  PIE_post
2       Amen Thompson    1610612745     19.2      26.5     0.134
8         Ayo Dosunmu    1610612741     25.5      37.5     0.092
20      Corey Kispert    1610612764     22.5      32.4     0.082
35         GG Jackson    1610612763     19.4      31.4     0.090
37        Gradey Dick    1610612761     15.3      28.8     0.060
63     Jordan Goodwin    1610612763     14.6      29.6     0.100
66  Julian Champagnie    1610612759     16.6      25.9     0.070
72        Kris Murray    1610612757     13.1      32.2     0.055


In [ ]:
# Merge following season net rating
final = merged_tanking.merge(
    following[['PLAYER_ID', 'PIE']],
    on='PLAYER_ID',
    how='left'
).rename(columns={'PIE': 'PIE_following'})

print(final[['PLAYER_NAME_post', 'PIE_post', 'PIE_pre', 'PIE_following']].to_string())

    PLAYER_NAME_post  PIE_post  PIE_pre  PIE_following
0      Amen Thompson     0.134    0.107          0.129
1        Ayo Dosunmu     0.092    0.077          0.084
2      Corey Kispert     0.082    0.085          0.074
3         GG Jackson     0.090    0.105          0.064
4        Gradey Dick     0.060    0.057          0.071
5     Jordan Goodwin     0.100    0.095          0.080
6  Julian Champagnie     0.070    0.066          0.079
7        Kris Murray     0.055    0.067          0.061


In [ ]:
def get_breakout_candidates(season):
    # Construct following season string
    start_year = int(season.split('-')[0])
    following_season_str = f"{start_year + 1}-{str(start_year + 2)[-2:]}"
    
    # Pull post all-star player stats
    post = leaguedashplayerstats.LeagueDashPlayerStats(
        season=season,
        season_segment_nullable='Post All-Star',
        measure_type_detailed_defense='Advanced'
    ).get_data_frames()[0]
    
    # Pull pre all-star player stats
    pre = leaguedashplayerstats.LeagueDashPlayerStats(
        season=season,
        season_segment_nullable='Pre All-Star',
        measure_type_detailed_defense='Advanced'
    ).get_data_frames()[0]
    
    # Pull following full season stats
    following = leaguedashplayerstats.LeagueDashPlayerStats(
        season=following_season_str,
        measure_type_detailed_defense='Advanced'
    ).get_data_frames()[0]
    
    # Pull pre all-star team standings
    pre_allstar_teams = leaguedashteamstats.LeagueDashTeamStats(
        season=season,
        season_segment_nullable='Pre All-Star'
    ).get_data_frames()[0]
    
    # Identify tanking teams (bottom 12 by pre all-star win rate)
    tanking_teams = pre_allstar_teams.nsmallest(12, 'W_PCT')[['TEAM_ID', 'TEAM_NAME']]
    
    # Keep relevant columns
    split_cols = ['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 
                  'AGE', 'GP', 'MIN', 'PIE', 'USG_PCT']
    post = post[split_cols].copy()
    pre = pre[split_cols].copy()
    following = following[['PLAYER_ID', 'PIE']].copy()
    
    # Apply filters
    post_filtered = post[
        (post['GP'] >= 15) &
        (post['MIN'] >= 25) &
        (post['AGE'] <= 28)
    ].copy()
    
    pre_filtered = pre[
        (pre['GP'] >= 20) &
        (pre['MIN'] >= 10)
    ].copy()
    
    # Merge pre and post
    merged = post_filtered.merge(
        pre_filtered,
        on='PLAYER_ID',
        suffixes=('_post', '_pre')
    )
    
    # Apply 30% minutes jump filter
    merged = merged[
        (merged['MIN_post'] - merged['MIN_pre']) / merged['MIN_pre'] >= 0.3
    ].copy()
    
    # Filter for tanking teams
    merged = merged[
        merged['TEAM_ID_post'].isin(tanking_teams['TEAM_ID'])
    ].copy()
    
    # Merge following season PIE
    merged = merged.merge(
        following[['PLAYER_ID', 'PIE']],
        on='PLAYER_ID',
        how='left'
    ).rename(columns={'PIE': 'PIE_following'})
    
    # Add season label
    merged['season'] = season
    
    return merged

print("Function defined successfully")